<a href="https://colab.research.google.com/github/CanerSivri/science-and-art/blob/Week-11/week11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import subprocess

def install_and_import(package):
    try:
        __import__(package)
        print(f"{package} is already installed.")
    except ImportError:
        print(f"Installing {package}...")
        subprocess.check_call(["pip", "install", "-q", package])
        print(f"{package} installed successfully.")

install_and_import("streamlit")
install_and_import("pyngrok")
install_and_import("transformers")
install_and_import("torch")
install_and_import("diffusers")

Installing streamlit...
streamlit installed successfully.
Installing pyngrok...
pyngrok installed successfully.
transformers is already installed.
torch is already installed.
diffusers is already installed.


In [2]:
from pyngrok import ngrok

NGROK_AUTH_TOKEN = "359sWI8TwzwDA9x1VmNt3ZgWLyM_5YXcrP1BW3qX6bGAscYv6"

if NGROK_AUTH_TOKEN == "YOUR_NGROK_AUTH_TOKEN":
    print("Please replace 'YOUR_NGROK_AUTH_TOKEN' with your actual ngrok authentication token.")
else:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    print("ngrok authentication token set.")

ngrok authentication token set.


In [3]:
%%writefile app.py
import streamlit as st
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import torch
import requests
from PIL import Image, ImageFilter, ImageEnhance
import io
from diffusers import StableDiffusionPipeline
import os # Import os for environment variables

st.set_page_config(page_title="Streamlit Chatbot & Art Generator", layout="centered")

st.sidebar.title("Select Mode")
selected_mode = st.sidebar.radio(
    "Choose a mode",
    ("Chat Mode", "Art Mode", "Image Filter Mode")
)


# --- Chat Mode Logic ---
if selected_mode == "Chat Mode":
    st.title("Chat Mode")
    st.write("Talk with a chatbot")

    @st.cache_resource
    def load_chat_model():
        tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
        model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")
        return tokenizer, model

    chat_tokenizer, chat_model = load_chat_model()

    if "chat_history" not in st.session_state:
        st.session_state.chat_history = []


    for i, message in enumerate(st.session_state.chat_history):
        if i % 2 == 0:
            st.text_input("You:", value=message, key=f"user_hist_{i}", disabled=True, label_visibility="collapsed")
        else:
            st.text_input("Bot:", value=message, key=f"bot_hist_{i}", disabled=True, label_visibility="collapsed")

    with st.form(key='chat_form', clear_on_submit=True):
        user_chat_input = st.text_input("Type your message here for chat:", key="user_chat_message_form_input")
        submitted = st.form_submit_button("Send")

        if submitted and user_chat_input:
            # Append user input to text history for display
            st.session_state.chat_history.append(user_chat_input)

            # Construct the full conversation context for Flan-T5
            formatted_conversation = ""
            # Iterate up to the current user input (last item in chat_history is the current user_chat_input)
            for i, msg in enumerate(st.session_state.chat_history):
                if i % 2 == 0:
                    formatted_conversation += f"User: {msg}\n"
                else:
                    formatted_conversation += f"Bot: {msg}\n"

            # For Flan-T5, we create a single prompt from the entire conversation history
            # The last line should imply the model needs to generate the bot's response
            prompt_text = formatted_conversation.strip() + "\nBot:"

            # Encode the prompt for the model
            input_ids = chat_tokenizer(prompt_text, return_tensors="pt").input_ids

            # Generate a response with sampling parameters
            with torch.no_grad():
                # For Seq2Seq models, `generate` directly produces the target sequence.
                output_ids = chat_model.generate(
                    input_ids,
                    max_length=500,
                    do_sample=True,
                    top_k=50,
                    top_p=0.95,
                    temperature=0.7,
                    num_return_sequences=1
                )

            # Decode the bot's response. For Seq2Seq models, no slicing is typically needed.
            bot_response = chat_tokenizer.decode(output_ids[0], skip_special_tokens=True)

            st.session_state.chat_history.append(bot_response)

            st.rerun()

# --- Art Mode Logic ---
elif selected_mode == "Art Mode":
    st.title("Art Mode")
    st.write("Generate images from text prompts using a locally loaded model.")
    st.warning("Generating image may take a while, please wait.")

    @st.cache_resource
    def load_art_model():

        hf_token = os.getenv('HF_TOKEN')

        if hf_token is None:
            st.warning("Hugging Face token not found in environment variables. Attempting to load model without explicit token, but it might fail for gated models like 'runwayml/stable-diffusion-v1-5'. Please ensure 'HF_TOKEN' is set in the environment where Streamlit is launched.")
            pipe = StableDiffusionPipeline.from_pretrained("runwayml/stable-diffusion-v1-5")
        else:
            pipe = StableDiffusionPipeline.from_pretrained("runwayml/stable-diffusion-v1-5", use_auth_token=hf_token)

        # Check for CUDA availability and move model accordingly
        if torch.cuda.is_available():
            pipe.to("cuda") # Move model to GPU if available
        else:
            pipe.to("cpu") # Fallback to CPU if no CUDA device is found
        return pipe

    art_pipeline = load_art_model()

    image_prompt = st.text_input("Enter your image prompt here:", key="image_prompt")

    if st.button("Generate Image"):
        if not image_prompt:
            st.error("Please enter a prompt to generate an image.")
        else:
            with st.spinner("Generating image..."):
                try:
                    # Generate image using the local pipeline
                    image = art_pipeline(image_prompt).images[0]
                    st.image(image, caption=image_prompt)
                except Exception as e:
                    st.error(f"Error generating image: {e}")

elif selected_mode == "Image Filter Mode":
    st.title("Image Filter Mode")
    st.write("Upload an image and apply various filters.")

    # Image filter functions
    def apply_grayscale(image):
        return image.convert("L")

    def apply_blur(image, radius=2):
        return image.filter(ImageFilter.GaussianBlur(radius))

    def apply_color_shift(image, r_factor=1.0, g_factor=1.0, b_factor=1.0):
        # Split into R, G, B bands
        r, g, b = image.split()
        # Apply enhancement factors
        r = ImageEnhance.Brightness(r).enhance(r_factor)
        g = ImageEnhance.Brightness(g).enhance(g_factor)
        b = ImageEnhance.Brightness(b).enhance(b_factor)
        # Merge back
        return Image.merge('RGB', (r, g, b))

    uploaded_file = st.file_uploader("Choose an image...", type=["jpg", "jpeg", "png"])

    if uploaded_file is not None:
        original_image = Image.open(uploaded_file).convert("RGB")
        st.subheader("Original Image")
        st.image(original_image, use_container_width=True)

        st.subheader("Apply Filters")
        selected_filter = st.selectbox(
            "Select a filter",
            ("None", "Grayscale", "Blur", "Color Shift")
        )

        filtered_image = None

        if selected_filter == "Grayscale":
            filtered_image = apply_grayscale(original_image)
        elif selected_filter == "Blur":
            blur_radius = st.slider("Blur Radius", 0.0, 10.0, 2.0, 0.1)
            filtered_image = apply_blur(original_image, blur_radius)
        elif selected_filter == "Color Shift":
            r_factor = st.slider("Red Channel Factor", 0.0, 2.0, 1.0, 0.05)
            g_factor = st.slider("Green Channel Factor", 0.0, 2.0, 1.0, 0.05)
            b_factor = st.slider("Blue Channel Factor", 0.0, 2.0, 1.0, 0.05)
            filtered_image = apply_color_shift(original_image, r_factor, g_factor, b_factor)

        if filtered_image:
            st.subheader(f"Filtered Image ({selected_filter})")
            st.image(filtered_image, use_container_width=True)
        elif selected_filter != "None":
            st.info("Select a filter to apply.")

Writing app.py


In [4]:
import subprocess
import time
from pyngrok import ngrok
from google.colab import userdata # Import userdata here
import os # Import os here

# Terminate existing Streamlit process and ngrok tunnels if they are active
if 'streamlit_process' in locals() and streamlit_process.poll() is None:
    streamlit_process.terminate()
    print("Terminated previous Streamlit process.")
ngrok.kill()
print("Terminated all ngrok tunnels.")

# Retrieve HF_TOKEN from Colab secrets
try:
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception as e:
    print(f"Could not retrieve Hugging Face token from Colab secrets: {e}")
    HF_TOKEN = None # Set to None if retrieval fails

# Prepare environment variables for the subprocess
# Copy current environment and add HF_TOKEN
env_vars = os.environ.copy()
if HF_TOKEN is not None:
    env_vars['HF_TOKEN'] = HF_TOKEN

# Run Streamlit app in the background
streamlit_process = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501", "--server.enableCORS", "false", "--server.enableXsrfProtection", "false"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    env=env_vars # Pass environment variables here
)

print("Streamlit chat and art app with mode selection started on port 8501.")

time.sleep(15)

# Create ngrok tunnel
tunnel = ngrok.connect(addr='8501', proto='http')
public_url = tunnel.public_url

print(f"Streamlit chat and art app is publicly accessible at: {public_url}")
print("You can interact with the chatbot and art generator by opening the URL in your browser.")
print("To stop the Streamlit app and ngrok tunnel later, run: streamlit_process.terminate() and ngrok.kill()")

Terminated all ngrok tunnels.
Streamlit chat and art app with mode selection started on port 8501.
Streamlit chat and art app is publicly accessible at: https://nether-eloy-agape.ngrok-free.dev
You can interact with the chatbot and art generator by opening the URL in your browser.
To stop the Streamlit app and ngrok tunnel later, run: streamlit_process.terminate() and ngrok.kill()
